# 서울 아파트 실거래가 첫 베이스라인 회귀 모델

이 노트북의 목적은 최고 성능 모델을 만드는 것이 아니라, 머신러닝 회귀 문제의 기본 흐름을 이해하는 것이다.

1. cleaned baseline 데이터 불러오기
2. 학습/검증 데이터 나누기
3. 평균값 기준 모델과 비교하기
4. 수치형 변수만 사용한 첫 회귀 모델 만들기
5. MAE/RMSE/R2를 억 원 단위로 해석하기

## 1. 라이브러리 불러오기

In [ ]:
from pathlib import Path
import warnings

import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager, rcParams

from sklearn.dummy import DummyRegressor
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore", category=RuntimeWarning)

FONT_PATH = "/System/Library/Fonts/AppleSDGothicNeo.ttc"
font_manager.fontManager.addfont(FONT_PATH)
FONT_NAME = font_manager.FontProperties(fname=FONT_PATH).get_name()
rcParams["font.family"] = FONT_NAME
rcParams["axes.unicode_minus"] = False
rcParams["figure.figsize"] = (9, 6)
rcParams["axes.grid"] = True

print(f"그래프 한글 폰트 설정 완료: {FONT_NAME}")

## 2. cleaned baseline 데이터 불러오기

In [ ]:
BASE_DIR = Path("..").resolve()
DATA_PATH = BASE_DIR / "data" / "processed" / "seoul_train_cleaned_baseline.csv"

df = pd.read_csv(DATA_PATH)
df["transaction_price_eok"] = df["transaction_real_price"] / 10000

df.shape

## 3. 사용할 변수 정하기

첫 모델에서는 위치 정보인 `dong`, `apt`를 아직 쓰지 않는다.

이유는 먼저 수치형 변수만으로 기준 성능을 만든 뒤, 범주형 변수를 추가했을 때 성능이 얼마나 좋아지는지 비교하기 위해서다.

In [ ]:
feature_cols = [
    "exclusive_use_area",
    "transaction_year_month",
    "floor",
    "building_age",
]
target_col = "transaction_real_price"

X = df[feature_cols]
y = df[target_col]

X.head()

## 4. 학습/검증 데이터 나누기

In [ ]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
)

X_train.shape, X_valid.shape

## 5. 모델 학습 및 비교

- 평균값 기준 모델: 아무 특징을 보지 않고 평균값만 예측하는 기준선
- 릿지 회귀 모델: 선형회귀의 안정적인 버전
- 수치형 부스팅 모델: 수치형 변수의 비선형 패턴을 잡는 모델

In [ ]:
models = {
    "평균값 기준 모델": DummyRegressor(strategy="mean"),
    "릿지 회귀 모델": make_pipeline(StandardScaler(), Ridge(alpha=1.0)),
    "수치형 부스팅 모델": HistGradientBoostingRegressor(
        max_iter=120,
        learning_rate=0.08,
        random_state=42,
    ),
}

rows = []
predictions = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_valid)
    predictions[name] = pred

    mae = mean_absolute_error(y_valid, pred)
    rmse = mean_squared_error(y_valid, pred) ** 0.5
    r2 = r2_score(y_valid, pred)

    rows.append({
        "모델": name,
        "MAE_만원": mae,
        "MAE_억 원": mae / 10000,
        "RMSE_만원": rmse,
        "RMSE_억 원": rmse / 10000,
        "R2": r2,
    })

metrics = pd.DataFrame(rows).sort_values("MAE_만원")
metrics.round(4)

## 6. 결과 해석

MAE는 평균적으로 예측이 실제 거래가에서 얼마나 벗어나는지를 뜻한다.

예를 들어 MAE가 `1.2억 원`이면, 평균적으로 실제 거래가와 예측 거래가가 약 1.2억 원 정도 차이 난다는 뜻이다.

In [ ]:
best_model_name = metrics.iloc[0]["모델"]
best_pred = predictions[best_model_name]

print("가장 좋은 첫 모델:", best_model_name)
print("MAE(억 원):", round(metrics.iloc[0]["MAE_억 원"], 4))

## 7. 실제 거래가 vs 예측 거래가

In [ ]:
plot_df = pd.DataFrame({
    "actual_eok": y_valid.values / 10000,
    "predicted_eok": best_pred / 10000,
}).sample(min(8000, len(y_valid)), random_state=42)

plt.figure(figsize=(7, 7))
plt.scatter(plot_df["actual_eok"], plot_df["predicted_eok"], s=7, alpha=0.28, color="#2563eb")

limit = min(25, max(plot_df["actual_eok"].max(), plot_df["predicted_eok"].max()))
plt.plot([0, limit], [0, limit], color="#ef4444", linewidth=2, label="실제값 = 예측값")
plt.xlim(0, limit)
plt.ylim(0, limit)
plt.title(f"{best_model_name}: 실제 거래가 vs 예측 거래가")
plt.xlabel("실제 거래가 (억 원)")
plt.ylabel("예측 거래가 (억 원)")
plt.legend()
plt.show()

## 8. 예측 오차 분포

In [ ]:
residual_eok = (y_valid.values - best_pred) / 10000
pd.Series(residual_eok).clip(-10, 10).hist(bins=80, color="#7c3aed")
plt.title(f"{best_model_name}: 예측 오차 분포 (-10억~10억 제한)")
plt.xlabel("오차 (실제 - 예측, 억 원)")
plt.ylabel("거래 건수")
plt.show()

## 9. 다음 단계

현재 모델은 위치 정보인 `dong`, `apt`를 사용하지 않았다.

서울 아파트 가격은 위치 영향이 크기 때문에, 다음 단계에서는 범주형 변수를 추가한 모델을 만들어 성능 변화를 비교한다.